# Module 7 Assignment: Rebuild Your Persona Agent on the OpenAI Agents SDK

Lab 6 rebuilt Study Buddy on top of the OpenAI Agents SDK: same persona, same notes, same three
tools from Lab 5, but with `Agent` and `Runner` replacing the hand written loop and dispatcher.

This assignment makes you do the same move on your **own** Persona Agent. By the end of Module 6
your agent had a persona, study notes, and three tools routed by a `call_tool(name, arguments)`
dispatcher you wrote by hand. Nothing about what your agent *does* needs to change here: you are
porting it to the framework, not redesigning it.

**What you'll build:**
1. An Azure configured `OpenAIChatCompletionsModel`, with tracing disabled, exactly like Lab 6
2. Your persona and notes, carried over from your Module 6 assignment
3. All three of your existing tools, each ported to `@function_tool`
4. One `Agent` that declares your persona, your notes, and all three tools
5. A test using `await Runner.run`
6. The same Gradio chat interface every lab and assignment has used since Lab 4, adapted from Lab 6

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 6 made the same kind of change to each of Study
Buddy's three tools, and do the analogous thing to yours.

## 0. Install the packages (run once)

Same as Lab 6.

In [1]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

Resolved 78 packages in 1.00s                                        
Prepared 72 packages in 10.57s                                               gradio                    ------------------------------ 29.76 MiB/29.91 MiB    gradio                    ------------------------------ 3.70 MiB/29.91 MiB     gradio                    ------------------------------ 3.30 MiB/29.91 MiB     gradio                    ------------------------------ 2.06 MiB/29.91 MiB     gradio                    ------------------------------ 1.98 MiB/29.91 MiB     gradio                    ------------------------------ 1.05 MiB/29.91 MiB     gradio                    ------------------------------ 784.00 KiB/29.91 MiB   gradio                    ------------------------------ 416.00 KiB/29.91 MiB   gradio                    ------------------------------ 352.00 KiB/29.91 MiB    gradio                    ------------------------------ 320.00 KiB/29.91 MiB   radio                    ---------------------------

## 1. Imports

**TODO:** Copy the imports from Lab 6, Step 1 (including `gradio as gr`), plus `requests` (and
anything else your own tools from the Module 6 assignment need, for example `html` if one of your
tools calls an API that returns HTML-escaped text).

In [2]:
from dotenv import load_dotenv
import os
import json

from openai import AsyncAzureOpenAI
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel, set_tracing_disabled
from pypdf import PdfReader
import gradio as gr
import requests

/Users/a111/Desktop/Assignment7_Maksimov_Denis/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load your Azure OpenAI credentials from `.env`

Identical to Lab 6 and every assignment before it.

In [3]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from Lab 6, Step 3, unchanged: call `set_tracing_disabled(True)`, build an
`AsyncAzureOpenAI` client from your endpoint, key, and API version, then wrap it and your deployment
name in `OpenAIChatCompletionsModel`.

In [4]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

## 4. Your persona and notes

**TODO:** Bring over your persona description and your study notes from your Module 6 assignment,
unchanged. If your notes lived in a PDF, reuse the same `load_notes` helper from Lab 6, Step 4.

In [5]:
persona = (
    "Professoressa Giulia, a warm and lively Italian teacher who sprinkles short Italian phrases "
    "into her replies (always with an English translation), celebrates right answers with "
    "\"Bravissimo!\", and corrects mistakes gently by showing the correct form and the rule behind it."
)


def load_notes(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text
    return text


notes = load_notes("notes/notes.pdf")

print(persona)
print(notes[:300], "...")

incorrect startxref pointer(1)
parsing for Object Streams


Professoressa Giulia, a warm and lively Italian teacher who sprinkles short Italian phrases into her replies (always with an English translation), celebrates right answers with "Bravissimo!", and corrects mistakes gently by showing the correct form and the rule behind it.
Italian Present Tense (Il Presente Indicativo)
The present tense in Italian describes what is happening now, habits, and near-future plans. Italian
verbs are grouped by their infinitive ending: -are, -ere, or -ire. To conjugate a regular verb, drop the
ending to get the stem, then add the endings fo ...


## 5. Port your first tool

**TODO:** Take your first tool from the Module 6 assignment and rewrite it as a plain, typed Python
function decorated with `@function_tool`, the same way Lab 6, Step 5a ported
`log_missed_question_tool`. Keep the function's actual behavior identical: only the decorator and
the removal of your hand written JSON schema should change.

In [6]:
@function_tool
def add_to_ripasso_tool(question: str) -> str:
    """Save a quiz question the student answered incorrectly to their ripasso (review) list.

    Args:
        question: The full text of the question the student got wrong.
    """
    print(f"Adding to your ripasso list: {question}")
    with open("ripasso.txt", "a", encoding="utf-8") as f:
        f.write(question + "\n")
    return f"Saved to ripasso.txt for later review: {question}"

## 6. Port your second tool

**TODO:** Same idea, for your second tool (in Module 6 this was the one backed by a free, no-key
external API), the same way Lab 6, Step 5b ported `fetch_bonus_question_tool`.

In [7]:
@function_tool
def translate_word_tool(word: str) -> str:
    """Look up the English meaning of an Italian word or short phrase using the free MyMemory API.

    Args:
        word: The Italian word or short phrase to translate into English, for example 'mangiare'.
    """
    print(f"Looking up the meaning of: {word}")
    try:
        response = requests.get(
            "https://api.mymemory.translated.net/get",
            params={"q": word, "langpair": "it|en"},
            timeout=10,
        )
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException:
        return f"The dictionary is not available right now, so I could not look up '{word}'."

    translation = data.get("responseData", {}).get("translatedText", "")
    if data.get("responseStatus") != 200 or not translation:
        return f"I could not find a translation for '{word}' right now."

    return f"'{word}' means '{translation}' in English."

## 7. Port your third tool

**TODO:** Same idea, for your third, self-designed tool, the same way Lab 6, Step 5c ported
`check_progress_tool`.

In [8]:
@function_tool
def track_score_tool(correct: bool) -> str:
    """Update the student's running score after a graded quiz answer.

    Args:
        correct: True if the student's answer was correct, False if it was wrong.
    """
    try:
        with open("score.json", "r", encoding="utf-8") as f:
            score = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        score = {"correct": 0, "wrong": 0}

    if correct:
        score["correct"] += 1
    else:
        score["wrong"] += 1

    with open("score.json", "w", encoding="utf-8") as f:
        json.dump(score, f)

    total = score["correct"] + score["wrong"]
    print(f"Score updated: {score['correct']} right, {score['wrong']} wrong")
    return f"Score so far: {score['correct']} out of {total} correct."

## 8. Build your Agent

**TODO:** Build one `Agent`, the same way Lab 6, Step 6 built Study Buddy: a `name`, `instructions`
built from your persona and your notes (with the same rules your old system prompt had about when
to use each tool), a `tools` list containing all three of your ported tools, and `model=azure_model`.

Notice what is missing compared to your Module 6 assignment: no `call_tool(name, arguments)`
dispatcher, and no JSON schema for any of the three tools. `Agent` and `@function_tool` already
cover both.

In [9]:
instructions = f"""
## Your role
You are {persona}
You are quizzing a student on the Italian present tense, using only the study notes below.

## Context
These are the student's study notes:

{notes}

## Rules
- Ask exactly ONE question at a time, then wait for the student's answer before asking the next one.
- Every question must be answerable from the notes above. Mix question types: conjugating a verb
  for a given person, translating a short sentence, or spotting and fixing a mistake.
- After each answer, say clearly whether it was correct. If it was wrong, give the correct answer
  and the rule from the notes that explains it, then move on to a new question.
- Whenever the student gets a question wrong, call the add_to_ripasso_tool tool with the full
  question text so it is saved to their review list. If the student says they missed several
  questions, call the tool once for each question.
- After you grade every answer, right or wrong, call the track_score_tool tool once with correct
  set to true or false, and mention the updated score in your reply.
- When the student asks what an Italian word means or asks for a vocabulary hint, call the
  translate_word_tool tool with that word instead of guessing. Vocabulary help is the one exception
  to staying inside the notes.
- If the student asks about anything else that is not in the notes, say kindly that it is outside
  today's lesson and steer back to the quiz. Do not invent grammar rules that are not in the notes.
- Keep replies short and friendly: a few sentences, not a lecture.
"""

giulia_agent = Agent(
    name="Professoressa Giulia",
    instructions=instructions,
    tools=[add_to_ripasso_tool, translate_word_tool, track_score_tool],
    model=azure_model,
)

print(f"{giulia_agent.name} is ready with tools:", [tool.name for tool in giulia_agent.tools])

Professoressa Giulia is ready with tools: ['add_to_ripasso_tool', 'translate_word_tool', 'track_score_tool']


## 9. Test it: `await Runner.run`

**TODO:** Run your agent with `await Runner.run(your_agent, "...")`, the same way Lab 6, Step 7
tested Study Buddy. Pick a message that should trigger at least one of your three tools, and
confirm the tool's visible effect (a line appended to a file, a number that changed, or whatever
your tool does) actually happened.

Stick to `await Runner.run(...)` here, not `Runner.run_sync`: Jupyter notebooks already run their
own event loop in the background, and `Runner.run_sync` raises
`RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.` when
called from a notebook cell.

In [ ]:
result = await Runner.run(
    giulia_agent,
    "What does the Italian word 'cercare' mean? Also, I got the question "
    "'Conjugate finire for io' wrong earlier, please save it to my review list.",
)
print(result.final_output)

# Confirm the tool's visible effect actually happened
print("\n--- ripasso.txt ---")
if os.path.exists("ripasso.txt"):
    with open("ripasso.txt", "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("ripasso.txt was not created, so add_to_ripasso_tool did not run. Try a clearer message.")

## 10. Give it a real chat UI with Gradio

**TODO:** Adapt the `chat` function and `gr.ChatInterface` call from Lab 6, Step 8 to your own
agent: an `async def chat(message, history)` function that rebuilds `history` as a plain list of
`{"role": ..., "content": ...}` dictionaries, appends the new `message` as one more `user` turn,
passes that to `await Runner.run(your_agent, ...)`, and returns `result.final_output`, handed to
`gr.ChatInterface(chat).launch(inbrowser=True)`.

As in Lab 6, `gr.ChatInterface` accepts your async `chat` function directly, and its `history`
already comes in the shape `Runner.run` expects, so no extra conversion is needed.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [ ]:
async def chat(message, history):
    messages = [{"role": h["role"], "content": h["content"]} for h in history]
    messages.append({"role": "user", "content": message})
    result = await Runner.run(giulia_agent, messages)
    return result.final_output


gr.ChatInterface(chat).launch(inbrowser=True)

## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. Which parts of your Module 6 assignment did you copy over completely unchanged? Which parts,
   if any, needed a real change rather than just a different decorator?
2. Compare your old `call_tool(name, arguments)` dispatcher to what `Agent` and `Runner` do for you
   now. What exactly is `Runner.run` doing in place of that dispatcher and your `while`/`for` loop?
3. Your Module 6 system prompt included sentences telling the model when to use each tool. Did any
   of that wording need to change for the OpenAI Agents SDK, or did it carry over directly into
   `instructions`?
4. If you added a fourth tool to this agent, what would you need to write, and what would you not
   need to write, compared to adding a fourth tool by hand back in Module 6?

## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm all three of your ported tools are reachable, `tools=[...]` lists all three, and at least
  one tool call actually happened during your test in Step 9 or Step 10.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module7Assignment_JaneDoe.ipynb`, then submit
  it.

## My reflection

**1. What carried over unchanged, and what really changed.** The persona, the study notes PDF, and the rule wording all came over word for word. So did the body of all three tools: the file writes in `add_to_ripasso_tool` and `track_score_tool`, and the MyMemory request in `translate_word_tool` with its `try`/`except` around `requests.exceptions.RequestException`. The one real change was not the decorator itself but what it now depends on. In Module 6 the type hints and docstrings were optional, because the hand written JSON schemas told the model everything. With `@function_tool` the schema is generated from the function, so the type hints (`question: str`, `correct: bool`) and the `Args:` section of each docstring now do the work my JSON `description` fields used to do. A missing type hint or a vague docstring would now produce a worse schema. The other real change was the client: `AzureOpenAI` became `AsyncAzureOpenAI`, and every call became `await Runner.run(...)`, because the SDK is asynchronous.

**2. What Runner.run does instead of my dispatcher and loop.** In Module 6 I wrote all of it by hand: send the messages and the `tools` list, check whether `finish_reason` was `"tool_calls"`, loop over every tool call, parse the arguments with `json.loads`, pick the right Python function in `call_tool` with an `if`/`elif` chain, append each result as a `role: "tool"` message, and call the model again until it returned plain text. `Runner.run` does exactly that cycle. It sends the agent's instructions, the input and the generated tool schemas to the model, and when the model asks for a tool it looks the tool up by name in the agent's `tools` list (this replaces `call_tool`), parses and validates the arguments against the type hints, runs the function, feeds the result back, and repeats until there is a final answer in `result.final_output`. It also stops after a maximum number of turns, a safety limit my own `while` loop never had.

**3. Did the tool rules need rewording?** No. The rule sentences carried over into `instructions` unchanged, because `instructions` is still just the system prompt: the SDK sends it as the first message, exactly like my `system_prompt` variable did. The model still chooses tools by reading plain English rules plus each tool's name and description. The only thing that had to stay consistent was the tool names, and since the `@function_tool` names match the old function names, the sentences that mention `add_to_ripasso_tool`, `translate_word_tool` and `track_score_tool` still point at the right tools.

**4. Adding a fourth tool.** I would still write the Python function itself, with type hints and a docstring describing when to use it, decorate it with `@function_tool`, add it to the `tools=[...]` list, and add one rule sentence to `instructions`. What I would no longer write is the JSON schema dictionary, an extra entry in a separate `tools` list of schemas, or a new `elif` branch in `call_tool`. In Module 6, forgetting that `elif` silently returned "Unknown tool" to the model. Here the tool name and the function cannot drift apart, because the SDK builds both the schema and the routing from the same function.